# 信贷违约预测 · 主流程 Notebook

信贷违约预测项目端到端主流程：**数据切分 → 特征工程 → 模型训练（最终配置 丙）→ 评估 → 业务应用**。

**最终模型（丙）**：XGBoost，`max_depth=6` / `learning_rate=0.02` / `subsample=0.8` / `colsample_bytree=0.7` / `reg_lambda=30` / `min_child_weight=24` / `gamma=0`，迭代上限 3000，seed 42，峰值轮 2920。

与之配套的交付：`模型优化报告v1.docx`（第 5 阶段）、`调优参数记录.xlsx`、`credit_dashboard.py`（业务仪表板）。

## 0. 运行环境

- 依赖：`pandas` `numpy` `scikit-learn` `lightgbm` `xgboost` `pyarrow` `matplotlib`。
- 前置数据：`db/processed/credit_clean.parquet`（第 2 阶段清洗产物，800,000 × 49）；原始数据见 `项目资料/credit_data(1).csv`。
- 训练建议使用 GPU（`xgboost` `gpu_hist`）；CPU 亦可，但耗时更长。
- 本 notebook 的"重跑"单元格已注释：直接运行不会触发完整重训；取消注释并保障前置数据后即可端到端复现。

In [1]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

ROOT = Path(os.getcwd()).resolve().parents[1]   # 项目根目录
sys.path.insert(0, str(ROOT / 'code' / '提交版'))

RANDOM_STATE = 42
TARGET = 'isDefault'

# 最终配置（丙）：深6 / lr0.02 / sub0.8 / col0.7 / λ30 / mcw24 / γ0
FINAL_CONFIG = dict(
    max_depth=6, learning_rate=0.02, subsample=0.8, colsample_bytree=0.7,
    reg_lambda=30, min_child_weight=24, gamma=0,
    n_estimators=3000, objective='binary:logistic', eval_metric='auc',
    tree_method='gpu_hist', random_state=RANDOM_STATE,
)
print('项目根:', ROOT)
print('最终配置（丙）:', FINAL_CONFIG)

项目根: <项目根目录>
最终配置（丙）: {'max_depth': 6, 'learning_rate': 0.02, 'subsample': 0.8, 'colsample_bytree': 0.7, 'reg_lambda': 30, 'min_child_weight': 24, 'gamma': 0, 'n_estimators': 3000, 'objective': 'binary:logistic', 'eval_metric': 'auc', 'tree_method': 'gpu_hist', 'random_state': 42}


## 1. 数据切分

- **random 分层抽样**：训练 70% / 验证 20% / 测试 10%（约 56 万 / 16 万 / 8 万），整体违约率 19.95%。训练集与验证集用于调参与早停，测试集仅用于最终报告。
- **oot_y2013 时间切分**：训练窗 2013 年起，验证/测试为更晚时段，用于跨时间外推（OOT）稳定性检验。

In [2]:
sp = pd.read_csv(ROOT / 'outputs' / 'tables' / '04_split_stats.csv', encoding='utf-8-sig')
sp[['tag', 'n_features', 'train_n', 'valid_n', 'test_n',
    'train_def', 'valid_def', 'test_def']]

,tag,n_features,train_n,valid_n,test_n,train_def,valid_def,test_def
0,random,89,559999,160000,80001,19.9513,19.9513,19.9510
1,oot_y2013,86,504475,160000,80000,19.0632,24.4250,19.9425


## 2. 特征工程

模块 `feature_engineering.py` 实现：

1. 类别编码：序数编码、独热编码、K 折 OOF 目标编码（高基数列 `title` / `employmentTitle` / `postCode` / `regionCode`）。
2. 数值变换：强右偏列 `log1p`，连续列标准化。
3. 等频分箱、多项式特征、组合/交叉特征（如 `loan_to_income`、`installment_to_income`）。
4. 特征选择：过滤法（互信息）、包装法（RFE）、嵌入法（模型重要性），最终约 89 维（全特征集）。

In [3]:
import feature_engineering as fe

print('feature_engineering 关键函数:')
print([n for n in dir(fe) if n.startswith(('split', 'encode', 'create', 'fit',
      'apply', 'build', 'select', 'imp'))][:20])

# 完整特征工程（生成 credit_fe_*.parquet 与 04_*.csv，需前置数据）：
#   fe.main('random')
#   fe.main('oot_y2013')
print('完整重跑请取消上方注释（建议 GPU）。')

feature_engineering 关键函数:
['apply_binning', 'apply_scaler', 'apply_target_encoding', 'build_and_ablate', 'build_combo_features', 'build_polynomial', 'create_features', 'encode_onehot', 'encode_ordinal', 'fit_binning', 'fit_scaler', 'fit_target_encoding', 'importance_analysis', 'select_embedded', 'select_filter', 'select_wrapper', 'selection_compare_figure', 'split_data']
完整重跑请取消上方注释（建议 GPU）。


## 3. 模型训练（最终配置 丙）

以 丙 配置在「训练集 + 验证集」合并样本上训练 XGBoost，早停 / 峰值轮约 2920。模块 `model_training.py` 封装训练、评估与所有结果表产出。

In [4]:
from xgboost import XGBClassifier

model = XGBClassifier(**FINAL_CONFIG)
model

# 完整训练与评估（生成 43_* / 50_* / 51_* / 76_* / 99_* 等结果表）：
#   import model_training as mt
#   mt.main(['random', 'oot_y2013'])
print('完整重跑请取消上方注释（建议 GPU）。')

完整重跑请取消上方注释（建议 GPU）。


## 4. 模型评估（权威结果）

以下直接读取 `outputs/tables` 中已产出的权威结果表，确保口径与《模型优化报告》一致，无需重新训练即可得到最终数字。

In [5]:
pe = pd.read_csv(ROOT / 'outputs' / 'tables' / '97_peak_round_choice.csv', encoding='utf-8-sig')
f = pe[(pe['切分'] == 'random') & (pe['特征集'] == '全特征') &
       (pe['配置'].str.startswith('丙')) & (pe['随机种子'] == 42)].iloc[0]

final_metrics = pd.DataFrame({
    '指标': ['测试 AUC', 'KS', '验证 AUC', 'Brier', '前30%捕获率', '业务最优阈值', '峰值轮'],
    '丙（最终）': [f['测试 auc'], f['测试 ks'], f['验证 AUC'], f['测试 brier'],
                f['前30%捕获率'], f['业务最优阈值'], f['峰值轮']],
})
final_metrics

,指标,丙（最终）
0,测试 AUC,0.739472
1,KS,0.351353
2,验证 AUC,0.739844
3,Brier,0.140006
4,前30%捕获率,0.566130
5,业务最优阈值,0.170000
6,峰值轮,2920.000000


In [6]:
# 调优前后对比（测试集，97 表 random/全特征/seed42 口径）
base = pe[(pe['切分'] == 'random') & (pe['特征集'] == '全特征') &
          (pe['配置'].str.startswith('基线')) & (pe['随机种子'] == 42)].iloc[0]
cmp = pd.DataFrame({
    '指标': ['测试 AUC', 'KS', '验证 AUC', '前30%捕获率', '业务最优阈值'],
    '第4阶段默认超参': [base['测试 auc'], base['测试 ks'], base['验证 AUC'],
                     base['前30%捕获率'], base['业务最优阈值']],
    '第5阶段调优后（丙）': [f['测试 auc'], f['测试 ks'], f['验证 AUC'],
                        f['前30%捕获率'], f['业务最优阈值']],
})
cmp

,指标,第4阶段默认超参,第5阶段调优后（丙）
0,测试 AUC,0.737967,0.739472
1,KS,0.349811,0.351353
2,验证 AUC,0.737587,0.739844
3,前30%捕获率,0.565253,0.566130
4,业务最优阈值,0.170000,0.170000


In [7]:
# 校准对比（合并重训口径，43 表）
cal = pd.read_csv(ROOT / 'outputs' / 'tables' / '43_calibration_after_opt.csv', encoding='utf-8-sig')
cal

,模型,测试集 AUC,Brier,LogLoss,ECE_验证集,ECE_测试集,MCE_测试集
0,XGBoost_默认超参_第4阶段,0.7383,0.14021,0.43967,0.01285,0.00254,0.0048
1,XGBoost_调优后_第5阶段,0.7402,0.13983,0.43860,0.01803,0.00316,0.0072


## 5. 业务应用

- **阈值决策**：成本敏感阈值扫描（`76_threshold_scan.csv`）给出业务最优阈值约 0.17（漏放:误拒 = 5:1 示例假设下）。
- **风险分层**：十分位 Lift（`68_lift_decile.csv`），首档 Lift 约 2.49，前 30% 人群捕获约 56.6% 的违约。
- **业务仪表板**：`code/提交版/credit_dashboard.py`（Streamlit），交互式呈现上述全部内容，支持阈值滑块与成本假设调节。

In [8]:
lift = pd.read_csv(ROOT / 'outputs' / 'tables' / '68_lift_decile.csv', encoding='utf-8-sig')
ts = pd.read_csv(ROOT / 'outputs' / 'tables' / '76_threshold_scan.csv', encoding='utf-8-sig')

lv5 = lift[lift['模型'] == '第5阶段调优后'].sort_values('十分位')
print('首档 Lift（调优后）: %.3f' % lv5['Lift'].iloc[0])
print('前30%%捕获率（97表 丙）: %.4f' % f['前30%捕获率'])
print('业务最优阈值附近的成本扫描:')
ts.iloc[(ts['阈值'] - 0.17).abs().argsort()[:3]]

首档 Lift（调优后）: 2.507
前30%捕获率（97表 丙）: 0.5661
业务最优阈值附近的成本扫描:


,阈值,拒绝率,精确率,召回率,F1,漏放数 FN,误拒数 FP,正确通过数 TN,YoudenJ,加权成本(5:1)
12,0.17,0.487919,0.312164,0.763423,0.443131,3776,26849,37191,0.344170,45729
13,0.18,0.460482,0.320204,0.739051,0.446818,4165,25043,38997,0.347999,45868
11,0.16,0.516131,0.303868,0.786104,0.438308,3414,28744,35296,0.337259,45814


## 6. 小结

- 最终模型（丙）较第 4 阶段默认超参，测试集 AUC 由 0.7380 提升至 0.7395，KS 由 0.3498 提升至 0.3514。
- 跨时间 OOT（训练窗 2013 起）测试 AUC 约 0.7226，较 random 切分下降约 0.017，属时间漂移预期范围，泛化稳定。
- 特征工程与正则化调优共同收窄了训练-测试间隙，模型可直接支撑审批阈值决策与风险分层，并通过业务仪表板落地。